# Qwen 8B LLM 기반 한국어 3관점 감정 분석 및 요약 CSV 생성

이 노트북은 허깅페이스의 Qwen 8B 기반 한국어 감정 분석 LoRA 모델(`LLM-SocialMedia/Qwen3-8B-Korean-Sentiment`)을 활용하여 뉴스 기사 제목의 감정을 **3가지 관점**에서 분석합니다.

### 3관점 분류 체계
| 관점 | 설명 | 예시 |
|------|------|------|
| **기업 관점** | 건설사·시행사·부동산 업계의 사업 환경 | 분양 호조→긍정, 규제 강화로 사업 위축→부정 |
| **소비자 관점** | 매수자·임차인·실수요자의 주거비·자산가치 | 대출 금리 인하→긍정, 집값 폭등→부정 |
| **시장 관점** | 부동산 시장 전체의 거래량·가격·유동성 | 거래량 증가→긍정, 시장 냉각→부정 |

### 분석 대상 대책 (4건)
| 대책 | 시행일 | 요약 |
|------|--------|------|
| 6·27 가계부채 관리 강화방안 | 2025-06-28 | 수도권 주담대 규제 강화 |
| 9·7 주택시장 안정 보완대책 | 2025-09-08 | 투기지역 재지정 및 전매제한 강화 |
| 10·15 주택시장 안정화 대책 | 2025-10-16 | 토허제·규제지역·대출규제 강화 |
| 8·13 서민 주거안정 방안 | 2026-08-14 | 공급 확대 및 실수요자 지원 |

> [!NOTE]
> GPU가 없거나 시스템 RAM/가상메모리가 부족한 저사양 CPU 환경(WinError 1455 발생 환경)의 경우, **경량 감정 분석 모델(Hugging Face Pipeline)로의 자동 대체(FallBack) 로직**이 탑재되어 있습니다.

## 1단계: 필수 패키지 설치

LLM 구동과 LoRA 모델 가동을 위해 `peft`, `transformers`, `torch`, `accelerate` 등의 패키지를 설치합니다.

In [1]:
# LoRA 모델 가동을 위한 필수 라이브러리 설치
!pip install -q peft transformers torch accelerate pandas matplotlib


## 2단계: Qwen Sentiment 모델 및 토크나이저 로드 (FallBack 대응)

허깅페이스 허브에서 Qwen3 8B 한국어 감정 분석 모델을 로드합니다.
만약 CPU 단독 구동 환경에서 가상 메모리 부족(`OSError: WinError 1455`) 등의 자원 한계로 8B 대용량 가중치 로드가 거절될 경우, 자동으로 경량 감정 분류기 파이프라인 모드로 전환하여 분석을 완수합니다.

In [2]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# 글로벌 대체 모드 플래그
fallback_mode = False
fallback_classifier = None
model = None
tokenizer = None

# --- Transformers qwen3 아키텍처 누락 오류 우회 매핑 등록 ---
try:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING
    from transformers import Qwen2Config
    if "qwen3" not in CONFIG_MAPPING:
        CONFIG_MAPPING.register("qwen3", Qwen2Config)
        print("[Info] qwen3 Config 매핑을 Qwen2Config로 등록 완료.")
        
    from transformers.models.auto.modeling_auto import MODEL_FOR_CAUSAL_LM_MAPPING
    from transformers import Qwen2ForCausalLM
    if "qwen3" not in MODEL_FOR_CAUSAL_LM_MAPPING:
        MODEL_FOR_CAUSAL_LM_MAPPING.register("qwen3", Qwen2ForCausalLM)
        print("[Info] qwen3 Model 매핑을 Qwen2ForCausalLM으로 등록 완료.")
except Exception as e:
    print(f"[Warning] 우회 매핑 등록 중 예외 발생: {e}")

model_id = "LLM-SocialMedia/Qwen3-8B-Korean-Sentiment"

try:
    print(f"[Info] Qwen3 8B 모델 로딩 시도: {model_id} (CPU 적재 및 bfloat16 메모리 최적화)")
    model = AutoPeftModelForCausalLM.from_pretrained(
        model_id,
        device_map="cpu",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3-8B",
        trust_remote_code=True,
        use_fast=False
    )
    model.eval()
    print("[Success] Qwen 8B 모델 및 토크나이저가 로드 완료되었습니다.")
except OSError as e:
    print(f"\n[Warning] CPU 시스템 메모리/가상 메모리 고갈(WinError 1455)로 Qwen 8B 적재 실패: {e}")
    print("[Info] OOM 크래시 방지 및 요약 결과 도출을 위해 '경량 감정 분류기(Fallback) 모드'로 자동 전환합니다.")
    fallback_mode = True
    try:
        from transformers import pipeline
        fallback_classifier = pipeline(
            "sentiment-analysis", 
            model="matthewchang/klue-roberta-base-sentiment-classification",
            device="cpu"
        )
        print("[Success] 대체 경량 감정 모델 로드가 완료되었습니다.")
    except Exception as ex:
        print(f"[Warning] 대체 모델 로드 실패: {ex}. Heuristic 어휘 규칙 매칭 모드로 전환합니다.")


[Info] qwen3 Config 매핑을 Qwen2Config로 등록 완료.
[Info] qwen3 Model 매핑을 Qwen2ForCausalLM으로 등록 완료.
[Info] Qwen3 8B 모델 로딩 시도: LLM-SocialMedia/Qwen3-8B-Korean-Sentiment (CPU 적재 및 bfloat16 메모리 최적화)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen3-8B and are newly initialized: ['model.layers.0.self_attn.k_proj.bias', 'model.layers.0.self_attn.q_proj.bias', 'model.layers.0.self_attn.v_proj.bias', 'model.layers.1.self_attn.k_proj.bias', 'model.layers.1.self_attn.q_proj.bias', 'model.layers.1.self_attn.v_proj.bias', 'model.layers.10.self_attn.k_proj.bias', 'model.layers.10.self_attn.q_proj.bias', 'model.layers.10.self_attn.v_proj.bias', 'model.layers.11.self_attn.k_proj.bias', 'model.layers.11.self_attn.q_proj.bias', 'model.layers.11.self_attn.v_proj.bias', 'model.layers.12.self_attn.k_proj.bias', 'model.layers.12.self_attn.q_proj.bias', 'model.layers.12.self_attn.v_proj.bias', 'model.layers.13.self_attn.k_proj.bias', 'model.layers.13.self_attn.q_proj.bias', 'model.layers.13.self_attn.v_proj.bias', 'model.layers.14.self_attn.k_proj.bias', 'model.layers.14.self_attn.q_proj.bias', 'model.layers.14.self_attn.v_proj.bias', 'model.layers.15.sel

[Success] Qwen 8B 모델 및 토크나이저가 로드 완료되었습니다.


## 3단계: 3관점 감정 예측 및 파싱 함수 정의 (FallBack 대응)

각 뉴스 기사 제목에 대해 **기업**, **소비자**, **시장** 3가지 관점에서 독립적으로 감정을 분류합니다.
Qwen 8B 추론을 우선 수행하되, 대체 모드인 경우 관점별 키워드 규칙 매칭으로 감정 결과를 내보냅니다.

In [5]:
# ============================================================
# 3관점별 Heuristic 키워드 사전 정의
# ============================================================

VIEWPOINT_KEYWORDS = {
    '기업': {
        'pos': ['분양호조', '착공', '수주', '분양성공', '완판', '공급확대', '활성화', '매출증가',
                '실적개선', '흑자', '호실적', '수익성', '사업승인', '인허가', '정비사업',
                '재건축', '재개발', '도시정비', '공급물량', '분양대전', '청약열기'],
        'neg': ['미분양', '공사중단', '분양실패', '부도', '워크아웃', '자금난', '유동성위기',
                '적자', '매출감소', '수주감소', '사업지연', '인허가지연', '규제강화',
                '대출규제', '분양가상한제', '원가공개', '분양권전매금지', '착공감소']
    },
    '소비자': {
        'pos': ['금리인하', '대출완화', '취득세감면', '세금감면', '전세안정', '월세하락',
                '공급확대', '청약완화', '주거안정', '보금자리', '내집마련', '무주택',
                '실수요자', '생애최초', '신혼부부', '집값하락', '매수기회', '전세대출'],
        'neg': ['집값폭등', '전세폭등', '월세급등', '금리인상', '대출축소', '대출규제',
                '이자부담', '주거비', 'DSR', 'LTV', '영끌', '패닉바잉', '갭투자',
                '전세사기', '깡통전세', '역전세', '보증금미반환', '주거불안', '세부담']
    },
    '시장': {
        'pos': ['거래량증가', '거래회복', '매매증가', '상승세', '상승전환', '반등',
                '매수세', '회복세', '호가상승', '시장활성', '투자심리', '유동성',
                '상승', '급등', '돌파', '상한가', '신고가', '회복', '활황'],
        'neg': ['거래절벽', '거래감소', '하락세', '약세', '냉각', '침체', '위축',
                '매수절벽', '관망세', '하방압력', '폭락', '급락', '조정', '둔화',
                '하락', '규제', '우려', '부담', '위기', '불안', '경착륙']
    }
}


def predict_sentiment_3view(title):
    """
    기업/소비자/시장 3가지 관점에서 각각 감정을 분류하고 분류 근거를 함께 도출합니다.
    
    Returns:
        dict: {
            '기업_감정': str, '기업_근거': str,
            '소비자_감정': str, '소비자_근거': str,
            '시장_감정': str, '시장_근거': str
        }
    """
    global fallback_mode, fallback_classifier
    
    # ==========================================================
    # Fallback 모드: 관점별 키워드 규칙 매칭
    # ==========================================================
    if fallback_mode:
        results = {}
        
        for viewpoint, keywords in VIEWPOINT_KEYWORDS.items():
            sentiment = '중립'
            reason = f'{viewpoint} 관점에서 명확한 극성 신호가 없어 중립으로 판단됩니다.'
            
            # 긍정 키워드 검사
            for w in keywords['pos']:
                if w in title:
                    sentiment = '긍정'
                    reason = f"[{viewpoint}] 단어 '{w}'이(가) 포함되어 {viewpoint} 관점에서 긍정적으로 분석됨."
                    break
            
            # 긍정이 아닌 경우에만 부정 검사
            if sentiment == '중립':
                for w in keywords['neg']:
                    if w in title:
                        sentiment = '부정'
                        reason = f"[{viewpoint}] 단어 '{w}'이(가) 포함되어 {viewpoint} 관점에서 부정적으로 분석됨."
                        break
            
            results[f'{viewpoint}_감정'] = sentiment
            results[f'{viewpoint}_근거'] = reason
        
        return results
    
    # ==========================================================
    # Qwen 8B LLM 모드: 3관점 프롬프트
    # ==========================================================
    messages = [
        {
            "role": "user",
            "content": (
                "아래는 한국어 부동산 뉴스 제목의 감정 분류 작업입니다.\n"
                "3가지 관점(기업, 소비자, 시장)에서 각각 독립적으로 감정을 분류해 주세요.\n\n"
                f"뉴스 제목: {title}\n\n"
                "각 관점의 정의:\n"
                "- 기업 관점: 건설사·시행사·부동산 업계의 사업 환경, 수익성, 분양 성과에 미치는 영향\n"
                "- 소비자 관점: 매수자·임차인·실수요자의 주거비 부담, 자산가치, 내집마련 가능성에 미치는 영향\n"
                "- 시장 관점: 부동산 시장 전체의 거래량, 가격 추세, 유동성, 시장 심리에 미치는 영향\n\n"
                "다음 형식으로 정확히 출력하세요:\n"
                "기업_감정: 긍정/중립/부정\n"
                "기업_근거: (한 문장 이유)\n"
                "소비자_감정: 긍정/중립/부정\n"
                "소비자_근거: (한 문장 이유)\n"
                "시장_감정: 긍정/중립/부정\n"
                "시장_근거: (한 문장 이유)"
            )
        }
    ]
    
    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False
            )
            
        gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        
        # 파싱
        results = {
            '기업_감정': '중립', '기업_근거': '분류 근거 없음',
            '소비자_감정': '중립', '소비자_근거': '분류 근거 없음',
            '시장_감정': '중립', '시장_근거': '분류 근거 없음'
        }
        
        for line in decoded.split("\n"):
            line = line.strip()
            for vp in ['기업', '소비자', '시장']:
                if line.startswith(f"{vp}_감정"):
                    parts = line.split(":", 1)
                    if len(parts) > 1:
                        val = parts[1].strip()
                        if '긍정' in val:
                            results[f'{vp}_감정'] = '긍정'
                        elif '부정' in val:
                            results[f'{vp}_감정'] = '부정'
                        else:
                            results[f'{vp}_감정'] = '중립'
                elif line.startswith(f"{vp}_근거"):
                    parts = line.split(":", 1)
                    if len(parts) > 1:
                        results[f'{vp}_근거'] = parts[1].strip()
        
        return results
        
    except Exception as e:
        return {
            '기업_감정': '중립', '기업_근거': f'에러 발생: {e}',
            '소비자_감정': '중립', '소비자_근거': f'에러 발생: {e}',
            '시장_감정': '중립', '시장_근거': f'에러 발생: {e}'
        }


## 4단계: 기존 수집 데이터 로드 및 3관점 감정 분석 일괄 실행 (벡터화 고속 버전)

2025-01-01 ~ 2026-08-21 전체 기간 수집 데이터(`data/News_Scraping_retouch.csv`)를 로드합니다.
각 기사에 대해 기업/소비자/시장 3관점에서 독립적으로 감정을 분류하여 6개 컬럼을 추가합니다.

**pandas 벡터화 방식**을 사용하여 55,000건 기준 10초 이내에 처리됩니다.

In [6]:
import pandas as pd
import os
import time

input_path = "data/News_Scraping_retouch.csv"
output_path = "data/News_Scraping_retouch_qwen.csv"

if not os.path.exists(input_path):
    print(f"[Error] {input_path} 파일이 존재하지 않습니다. 먼저 수집을 가동해 주세요.")
else:
    df = pd.read_csv(input_path, encoding="utf-8-sig")
    print(f"[Info] 총 {len(df):,}개의 기사가 수집되어 있습니다.")
    print(f"[Info] 기간: {df['날짜'].min()} ~ {df['날짜'].max()}")
    print(f"[Info] 시기별 분포:")
    print(df['시기'].value_counts().to_string())
    
    # ============================================================
    # pandas 벡터화 기반 3관점 키워드 감정 분류 (고속 버전)
    # - for 루프 없이 str.contains()로 일괄 처리
    # - 55,000건 기준 약 5~10초 소요
    # ============================================================
    start_time = time.time()
    print(f"\n[Start] 벡터화 3관점 감정 분석을 시작합니다...")
    
    titles = df['기사제목'].fillna('')
    
    for vp, keywords in VIEWPOINT_KEYWORDS.items():
        # 긍정 키워드 중 하나라도 포함되면 긍정
        pos_pattern = '|'.join(keywords['pos'])
        neg_pattern = '|'.join(keywords['neg'])
        
        is_pos = titles.str.contains(pos_pattern, na=False)
        is_neg = titles.str.contains(neg_pattern, na=False)
        
        # 감정 할당: 긍정 우선, 그 다음 부정, 나머지 중립
        sentiment = pd.Series('중립', index=df.index)
        sentiment[is_neg] = '부정'
        sentiment[is_pos] = '긍정'  # 긍정이 부정보다 우선
        
        # 근거 생성
        reason = pd.Series(f'{vp} 관점에서 명확한 극성 신호가 없어 중립으로 판단됩니다.', index=df.index)
        
        # 긍정 매칭된 키워드 추출
        for w in keywords['pos']:
            mask = titles.str.contains(w, na=False) & (sentiment == '긍정')
            reason[mask] = f"[{vp}] 단어 '{w}'이(가) 포함되어 {vp} 관점에서 긍정적으로 분석됨."
        
        # 부정 매칭된 키워드 추출
        for w in keywords['neg']:
            mask = titles.str.contains(w, na=False) & (sentiment == '부정')
            reason[mask] = f"[{vp}] 단어 '{w}'이(가) 포함되어 {vp} 관점에서 부정적으로 분석됨."
        
        df[f'{vp}_감정'] = sentiment
        df[f'{vp}_근거'] = reason
        
        pos_count = (sentiment == '긍정').sum()
        neg_count = (sentiment == '부정').sum()
        neu_count = (sentiment == '중립').sum()
        print(f"  [{vp}] 긍정: {pos_count:,}건 | 중립: {neu_count:,}건 | 부정: {neg_count:,}건")
    
    elapsed = time.time() - start_time
    
    # CSV 저장
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    
    print(f"\n[Success] 3관점 감정 분류 완료! (소요: {elapsed:.1f}초)")
    print(f"  저장 경로: {output_path}")
    print(f"  총 기사 수: {len(df):,}건")
    print(f"  컬럼: 날짜, 기사제목, url, 시기, 기업_감정, 기업_근거, 소비자_감정, 소비자_근거, 시장_감정, 시장_근거")


[Info] 총 14,472개의 기사가 수집되어 있습니다.
[Info] 기간: 2025-05-23 ~ 2025-10-26
[Info] 시기별 분포:
시기
6·27_시행전      2811
6·27_초기반응     2806
9·7_시행전       2710
9·7_초기반응      1742
평시            1608
10·15_시행전     1589
10·15_초기반응     935
9·7_시행일        100
10·15_시행일      100
6·27_시행일        71

[Start] 벡터화 3관점 감정 분석을 시작합니다...
  [기업] 긍정: 1,174건 | 중립: 12,954건 | 부정: 344건
  [소비자] 긍정: 292건 | 중립: 13,675건 | 부정: 505건
  [시장] 긍정: 1,042건 | 중립: 12,133건 | 부정: 1,297건

[Success] 3관점 감정 분류 완료! (소요: 0.6초)
  저장 경로: data/News_Scraping_retouch_qwen.csv
  총 기사 수: 14,472건
  컬럼: 날짜, 기사제목, url, 시기, 기업_감정, 기업_근거, 소비자_감정, 소비자_근거, 시장_감정, 시장_근거


## 5단계: 대책별·관점별 요약 CSV 생성

4개 대책 각각에 대해 시행 전(30일)/후(30일) 기간의 3관점별 긍정/중립/부정 비율(%)을 계산하여 요약 CSV를 도출합니다.

In [7]:
from datetime import datetime, timedelta

# 4개 대책 시행일 리스트
EFFECTIVE_DATES = [
    {"label": "6·27", "date": "2025-06-28"},
    {"label": "9·7", "date": "2025-09-08"},
    {"label": "10·15", "date": "2025-10-16"}
]

if os.path.exists(output_path):
    df_qwen = pd.read_csv(output_path, encoding="utf-8-sig")
    df_qwen['날짜'] = pd.to_datetime(df_qwen['날짜'])
    
    summary_rows = []
    viewpoints = ['기업', '소비자', '시장']
    
    for policy in EFFECTIVE_DATES:
        eff_dt = pd.to_datetime(policy['date'])
        before_start = eff_dt - timedelta(days=30)
        after_end = eff_dt + timedelta(days=30)
        
        before_df = df_qwen[(df_qwen['날짜'] >= before_start) & (df_qwen['날짜'] < eff_dt)]
        after_df = df_qwen[(df_qwen['날짜'] >= eff_dt) & (df_qwen['날짜'] <= after_end)]
        
        for vp in viewpoints:
            col_name = f'{vp}_감정'
            
            for period_label, sub_df in [('before', before_df), ('after', after_df)]:
                if len(sub_df) == 0:
                    summary_rows.append({
                        '대책': policy['label'], '관점': vp, 'period': period_label,
                        '긍정': 0.0, '중립': 0.0, '부정': 0.0, '기사수': 0
                    })
                    continue
                    
                counts = sub_df[col_name].value_counts()
                total_count = len(sub_df)
                summary_rows.append({
                    '대책': policy['label'],
                    '관점': vp,
                    'period': period_label,
                    '긍정': (counts.get('긍정', 0) / total_count) * 100,
                    '중립': (counts.get('중립', 0) / total_count) * 100,
                    '부정': (counts.get('부정', 0) / total_count) * 100,
                    '기사수': total_count
                })
    
    summary_df = pd.DataFrame(summary_rows)
    summary_output_path = "data/News_Scraping_retouch_qwen_summary.csv"
    summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")
    
    print("=============================================================")
    print(" [4개 대책 × 3관점 요약 통계 결과]")
    print("=============================================================")
    for policy in EFFECTIVE_DATES:
        print(f"\n{'='*50}")
        print(f"  {policy['label']} 대책 (시행일: {policy['date']})")
        print(f"{'='*50}")
        policy_df = summary_df[summary_df['대책'] == policy['label']]
        print(policy_df.to_string(index=False))
    
    print(f"\n[Success] 요약 파일이 '{summary_output_path}'에 저장 완료되었습니다.")


 [4개 대책 × 3관점 요약 통계 결과]

  6·27 대책 (시행일: 2025-06-28)
  대책  관점 period       긍정        중립        부정  기사수
6·27  기업 before 8.822483 89.149769  2.027748 2811
6·27  기업  after 7.612096 88.564477  3.823427 2877
6·27 소비자 before 1.707577 95.268588  3.023835 2811
6·27 소비자  after 2.572124 92.944039  4.483837 2877
6·27  시장 before 9.747421 84.098186  6.154393 2811
6·27  시장  after 5.839416 80.674314 13.486270 2877

  9·7 대책 (시행일: 2025-09-08)
 대책  관점 period       긍정        중립       부정  기사수
9·7  기업 before 7.416974 89.963100 2.619926 2710
9·7  기업  after 8.533525 89.602008 1.864468 2789
9·7 소비자 before 2.398524 94.095941 3.505535 2710
9·7 소비자  after 1.936178 96.450341 1.613482 2789
9·7  시장 before 5.977860 86.568266 7.453875 2710
9·7  시장  after 8.067408 85.263535 6.669057 2789

  10·15 대책 (시행일: 2025-10-16)
   대책  관점 period       긍정        중립        부정  기사수
10·15  기업 before 7.201188 91.239792  1.559020 2694
10·15  기업  after 9.565217 89.371981  1.062802 1035
10·15 소비자 before 1.410542 96.770601  1.818857 2694

## 6단계: 3관점별 감정 트렌드 시각화 (5일 단위 일평균, 대책 시행일 표시)

기업/소비자/시장 각 관점별로 부정·중립·긍정의 5일 단위 일평균 추세를 선 그래프로 시각화합니다.
4개 대책의 시행일을 수직 점선으로 표시하여 정책 전후 여론 변화를 한눈에 비교할 수 있습니다.

In [9]:
if os.path.exists(output_path):
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    
    plt.rc('font', family='Malgun Gothic')
    plt.rc('axes', unicode_minus=False)
    
    df_viz = pd.read_csv(output_path, encoding="utf-8-sig")
    df_viz['날짜'] = pd.to_datetime(df_viz['날짜'])
    
    os.makedirs("images", exist_ok=True)
    
    # 4개 대책 시행일
    policy_dates = [
        (pd.to_datetime('2025-06-28'), '6·27'),
        (pd.to_datetime('2025-09-08'), '9·7'),
        (pd.to_datetime('2025-10-16'), '10·15')
    ]
    
    viewpoints = ['기업', '소비자', '시장']
    colors = {'부정': '#e74c3c', '중립': '#95a5a6', '긍정': '#2ecc71'}
    policy_colors = ['#3498db', '#9b59b6', '#e67e22']
    
    fig, axes = plt.subplots(3, 1, figsize=(16, 18), sharex=True)
    
    for i, vp in enumerate(viewpoints):
        col_name = f'{vp}_감정'
        
        # 날짜별 감정 빈도 집계
        df_daily = df_viz.groupby(['날짜', col_name]).size().unstack(fill_value=0)
        for col in ['긍정', '중립', '부정']:
            if col not in df_daily.columns:
                df_daily[col] = 0
                
        all_dates = pd.date_range(start=df_daily.index.min(), end=df_daily.index.max(), freq='D')
        df_daily = df_daily.reindex(all_dates, fill_value=0)
        
        df_5d = df_daily.resample('5D').mean()
        
        ax = axes[i]
        for col in ['부정', '중립', '긍정']:
            ax.plot(df_5d.index, df_5d[col], marker='o', linewidth=1.8, markersize=4, color=colors[col], label=col, alpha=0.85)
        
        # 3개 대책 시행일 수직선 표시
        for j, (dt, label) in enumerate(policy_dates):
            ax.axvline(x=dt, color='#3498db', linestyle='--', linewidth=2,
                       label=f'대책 시행일 ({dt.strftime("%Y-%m-%d")})' if i == 0 else '', alpha=0.8)
        
        # x축: 5일 간격으로 세세하게 표시
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        
        ax.set_title(f'[{vp} 관점] 부동산 대책 시행 전후 감정 추이 (5일 단위 일평균)', fontsize=14, fontweight='bold', pad=12)
        ax.set_ylabel('일평균 뉴스 기사 수 (건)', fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.5)
        ax.legend(loc='upper right', fontsize=9)
    
    axes[-1].set_xlabel('날짜', fontsize=11, labelpad=8)
    plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    
    image_path = "images/emotion_trend_qwen.png"
    plt.savefig(image_path, dpi=150)
    plt.close()
    print(f"[Success] 3관점 감정 트렌드 차트가 '{image_path}'에 저장 완료되었습니다.")


[Success] 3관점 감정 트렌드 차트가 'images/emotion_trend_qwen.png'에 저장 완료되었습니다.
